# Data Science Research Capstone & Whitepaper
## Financial Transaction Fraud Risk & Behavioral Pattern Analysis

**Author:** Amit Singh Bhadouriya  
**Project:** Data Science Research Capstone  

This notebook reproduces the end-to-end data pipeline, statistical hypothesis testing (Mann–Whitney U), predictive fraud classification (Logistic Regression, Random Forest), unsupervised customer clustering (K-Means), and behavioral risk visualization.

In [ ]:
import os
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import classification_report, roc_auc_score, silhouette_score

# Prevent KMeans memory leak UserWarning on Windows with MKL
os.environ["OMP_NUM_THREADS"] = "1"
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")
%matplotlib inline

In [ ]:
# Robust dataset resolution whether running from notebooks/ or workspace root
candidates = [
    Path('../data/capstone_demo_dataset.csv'),
    Path('data/capstone_demo_dataset.csv'),
    Path('data_science_capstone_whitepaper/data/capstone_demo_dataset.csv'),
    Path('capstone_demo_dataset.csv')
]

data_path = None
for p in candidates:
    if p.exists():
        data_path = p
        break

if data_path is None:
    raise FileNotFoundError("Could not locate capstone_demo_dataset.csv in expected paths.")

print(f"Loading dataset from: {data_path}")
df = pd.read_csv(data_path)
print(f"Total records: {len(df)} | Columns: {len(df.columns)}")
df.head()

In [ ]:
# Timestamp parsing and temporal feature extraction
df['timestamp'] = pd.to_datetime(df['transaction_date'] + ' ' + df['transaction_time'], errors='coerce')
df = df.dropna(subset=['timestamp', 'amount', 'is_fraud']).copy()
df['hour'] = df['timestamp'].dt.hour
df['is_night'] = ((df['hour'] < 6) | (df['hour'] >= 23)).astype(int)

fraud_count = int(df['is_fraud'].sum())
fraud_pct = df['is_fraud'].mean() * 100
print(f"Cleaned dataset size: {len(df)}")
print(f"Fraudulent transactions: {fraud_count} ({fraud_pct:.2f}%)")
df[['amount', 'previous_transactions', 'account_age_days', 'failed_attempts', 'hour', 'is_fraud']].describe()

In [ ]:
# Non-parametric Mann-Whitney U tests for Amount and Failed Attempts
fraud = df.loc[df['is_fraud'] == 1]
normal = df.loc[df['is_fraud'] == 0]

u_amt, p_amt = mannwhitneyu(fraud['amount'], normal['amount'], alternative='two-sided')
u_fail, p_fail = mannwhitneyu(fraud['failed_attempts'], normal['failed_attempts'], alternative='two-sided')

print("=== Statistical Hypothesis Testing (Mann-Whitney U) ===")
print(f"1. Transaction Amount: U = {u_amt}, p-value = {p_amt:.4e}")
print(f"   - Fraud median: {fraud['amount'].median():.2f} vs Normal median: {normal['amount'].median():.2f}")
print(f"2. Failed Attempts:   U = {u_fail}, p-value = {p_fail:.4e}")
print(f"   - Fraud median: {fraud['failed_attempts'].median():.2f} vs Normal median: {normal['failed_attempts'].median():.2f}")

In [ ]:
features = ['amount', 'previous_transactions', 'account_age_days', 'failed_attempts', 'hour']
X = df[features]
y = df['is_fraud'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

models = {
    'Logistic Regression': Pipeline([
        ('scale', StandardScaler()),
        ('model', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42))
    ]),
    'Random Forest': RandomForestClassifier(
        n_estimators=300, max_depth=6, class_weight='balanced', random_state=42
    )
}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    
    print(f"\n==================== {name} ====================")
    print(f"ROC-AUC: {roc_auc_score(y_test, proba):.4f}")
    print("Classification Report:")
    print(classification_report(y_test, preds, zero_division=0))

In [ ]:
# K-Means clustering with standardized features
scaler = StandardScaler()
Z = scaler.fit_transform(df[features])

km = KMeans(n_clusters=3, n_init=20, random_state=42)
df['cluster'] = km.fit_predict(Z)
sil = silhouette_score(Z, df['cluster'])

print(f"K-Means Silhouette Score (k=3): {sil:.4f}")
print("\nCluster Profile Summary:")
cluster_summary = df.groupby('cluster').agg(
    transactions=('transaction_id', 'count'),
    fraud_rate=('is_fraud', 'mean'),
    avg_amount=('amount', 'mean'),
    median_amount=('amount', 'median'),
    avg_failed=('failed_attempts', 'mean')
).reset_index()
cluster_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Plot 1: Transaction Amount by Fraud Status
fraud_amounts = [df.loc[df['is_fraud'] == 0, 'amount'], df.loc[df['is_fraud'] == 1, 'amount']]
axes[0].boxplot(fraud_amounts, tick_labels=['Legitimate (0)', 'Fraudulent (1)'], patch_artist=True)
axes[0].set_title('Transaction Amount by Fraud Status', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Amount')
axes[0].grid(True, alpha=0.3)

# Plot 2: Fraud Rate by Cluster
colors = ['#e74c3c', '#3498db', '#2ecc71']
axes[1].bar(cluster_summary['cluster'].astype(str), cluster_summary['fraud_rate'] * 100, color=colors[:len(cluster_summary)])
axes[1].set_title('Fraud Rate by K-Means Cluster (%)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Cluster ID')
axes[1].set_ylabel('Fraud Rate (%)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()